# Capstone: Your Own Chatbot

See also: [project brief](https://claude.ai/artifact/8S47LsiuvRYk52FksB8T4t)

# Your name(s) - AI-use statement

Name(s): ...

AI use: (tool) - (what for: which function or part) - (what you
checked yourself). Or: none. In a pair: who wrote which extension.

# How to use this notebook

1. `Runtime > Run all`. The last cell starts a chat; type `bye` to stop.
2. Change the persona cell: your own rules and replies.
3. After every change, run all cells again and read the check cell.
4. To hand in: `File > Download > Download .ipynb`, upload to Canvas.

Put every pattern and every reply in single quotes `'...'`.

# 1. Your persona

Running this cell writes the file. Rename it to `yourname.yaml` - here
and in the next cell.

In [ ]:
%%writefile doctor.yaml
# DOCTOR - a persona for the ELIZA engine, in the style of
# Weizenbaum's 1966 script. The replies are written for this course.
#
# The engine tries the rules from top to bottom and uses the first
# rule whose pattern matches. Put specific rules above general ones.
# In a reply, {x} stands for group 1 of the pattern, reflected
# ("my" becomes "your").

name: ELIZA
greeting: How do you do. What would you like to talk about?
goodbye: Goodbye. Thank you for talking to me.

# Typing one of these ends the conversation.
quit:
  - bye
  - goodbye
  - quit

# Used when no rule matches, one after the other.
fallback:
  - Please go on.
  - Tell me more.
  - What does that suggest to you?
  - I see. Can you say more about that?

# Swapped word by word in {x}.
reflections:
  i: you
  me: you
  my: your
  am: are
  mine: yours
  myself: yourself
  you: I
  your: my
  yours: mine
  are: am

rules:
  - pattern: '^i am (.*)'
    replies:
      - Why are you {x}?
      - How long have you been {x}?
      - Do you enjoy being {x}?

  - pattern: '^i feel (.*)'
    replies:
      - Do you often feel {x}?
      - When did you start feeling {x}?

  - pattern: '\bi need (.*)'
    replies:
      - Why do you need {x}?
      - Would it really help you to get {x}?

  - pattern: '\bi want (.*)'
    replies:
      - Why do you want {x}?
      - Suppose you got {x}. What then?

  - pattern: '\bmy (mother|father|sister|brother)\b'
    replies:
      - Tell me more about your {x}.
      - How do you get along with your {x}?

  - pattern: '\bbecause\b(.*)'
    replies:
      - Is that the real reason?
      - What other reasons come to mind?

  - pattern: '\b(always|never)\b'
    replies:
      - Can you think of a specific example?
      - Really, {x}?

  - pattern: '\b(dream|dreams|dreamed)\b'
    replies:
      - What does that dream suggest to you?
      - Do you dream often?

  - pattern: '\b(computer|computers)\b'
    replies:
      - Do computers worry you?
      - What do you think about machines that talk?

  - pattern: '\bsorry\b(.*)'
    replies:
      - Please don't apologize.
      - Apologies are not necessary.

In [ ]:
SCRIPT = "doctor.yaml"   # the persona the engine loads

# 2. The engine

Given. Change `respond` and `main` only to call your extension(s). You don't
need to understand this cell yet: by the time we get to dictionaries,
you will.

In [ ]:
import re
import yaml

LOG = "log.txt"   # every exchange is appended here


def load_script(filename):
    """Return the script in filename as a dictionary."""
    file = open(filename)
    script = yaml.safe_load(file)
    file.close()
    return script


def normalize(sentence):
    """Return sentence in lowercase, with "i'm"/"im" written out,
    without punctuation at the end.

    >>> normalize("I'm tired!")
    'i am tired'
    >>> normalize("Tell him im sorry.")
    'tell him i am sorry'
    """
    sentence = sentence.lower().strip()
    sentence = re.sub(r"\bi'?m\b", "i am", sentence)
    return sentence.strip(".!?")


def reflect(text, reflections):
    """Return text with each word swapped by the reflections dict.

    >>> reflect("my sister likes me", {"my": "your", "me": "you"})
    'your sister likes you'
    """
    words = []
    for word in text.split():
        words.append(reflections.get(word, word))
    return " ".join(words)


def next_reply(replies):
    """Return the first reply and move it to the end of the list.

    >>> replies = ["A", "B"]
    >>> next_reply(replies)
    'A'
    >>> next_reply(replies)
    'B'
    >>> next_reply(replies)
    'A'
    """
    reply = replies.pop(0)
    replies.append(reply)
    return reply


def respond(sentence, script):
    """Return the reply of the first rule whose pattern matches
    sentence, or the next fallback reply if no rule matches."""
    sentence = normalize(sentence)
    for rule in script["rules"]:
        match = re.search(rule["pattern"], sentence)
        if match is not None:
            reply = next_reply(rule["replies"])
            if "{x}" in reply:
                rest = reflect(match.group(1), script["reflections"])
                reply = reply.replace("{x}", rest)
            return reply
    return next_reply(script["fallback"])


def log_exchange(sentence, reply):
    """Append one exchange to the log file."""
    log = open(LOG, "a")
    log.write(f"You: {sentence}\n")
    log.write(f"{reply}\n")
    log.close()


def run_conversation(filename, script):
    """Print a numbered transcript for the user lines in filename."""
    number = 1
    for line in open(filename):
        sentence = line.strip()
        if sentence != "":
            print(f"{number}. You: {sentence}")
            print(f"   {script['name']}: {respond(sentence, script)}")
            number += 1


def main():
    """Talk to the user until they type one of the quit words."""
    script = load_script(SCRIPT)
    print(f"{script['name']}: {script['greeting']}")
    while True:
        sentence = input("You: ")
        if normalize(sentence) in script["quit"]:
            print(f"{script['name']}: {script['goodbye']}")
            break
        reply = respond(sentence, script)
        print(f"{script['name']}: {reply}")
        log_exchange(sentence, f"{script['name']}: {reply}")

def check(script, filename):
    """Print the counts the capstone asks for, and every rule that
    never fires for the test lines in filename."""
    rules = script["rules"]
    two = 0
    with_x = 0
    with_b = 0
    for rule in rules:
        if len(rule["replies"]) >= 2:
            two += 1
        if "{x}" in " ".join(rule["replies"]):
            with_x += 1
        if r"\b" in rule["pattern"]:
            with_b += 1
    print("Rules:", len(rules), "(need 12)")
    print("Rules with 2+ replies:", two, "(need all)")
    print("Rules with {x}:", with_x, "(need 4)")
    print("Rules with \\b:", with_b, "(need 3)")
    fired = {}
    for line in open(filename):
        sentence = normalize(line)
        for rule in rules:
            if re.search(rule["pattern"], sentence):
                fired[rule["pattern"]] = True
                break
    for rule in rules:
        if rule["pattern"] not in fired:
            print("Never fires:", rule["pattern"])

# 3. Your extension(s)

One for each of you, from the extension list or your own idea:
memory, statistics, your name, tone, Markov fallback.
Write it as new functions that the program uses, each with a
docstring and at least 2 doctests.

In [ ]:
# your extension's functions here

# 4. Doctests

Should end with =failed=0=.

In [ ]:
import doctest
doctest.testmod()

# 5. Test conversation

At least 15 lines; every rule fires at least once.

In [ ]:
%%writefile conversation.txt
Hello there.
I am tired of my homework.
I'm worried about my exams.
I feel nervous today.
My sister gets better grades.
Because she studies more than I do.
I always forget the dates.
I need a break.
I want to sleep for a week.
Computers make it worse.
Sorry, I am complaining a lot.
I had a strange dream last night.
The weather is nice.
I am tired again.

In [ ]:
script = load_script(SCRIPT)
run_conversation("conversation.txt", script)

# 6. Check

In [ ]:
check(load_script(SCRIPT), "conversation.txt")

# 7. Chat

Type `bye` to stop.

In [ ]:
main()